MEMO Debugging

In [1]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
# from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

# d, h, l = 1024, 4, 2
# chunk_length = 16
d,h,l = 2048, 4, 3
chunk_length = 4096

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
               hidden_size=d, 
               num_hidden_layers=l,
               num_attention_heads=h,
               chunk_length=chunk_length,
               bos_token_id=tokenizer.bos_token_id,
               eos_token_id=tokenizer.eos_token_id,
               pad_token_id=tokenizer.pad_token_id,
              )

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.


Setting pad token and pad token id = <|endoftext|>, 0
MeMo embedding initilialization
MeMo embedding initilialization
MeMo embedding initilialization
GPU: NVIDIA RTX A6000 is available.


In [2]:
# evaluation method
from MeMoHF.evaluating_memo import EvaluationUpdateNew
evaluation = EvaluationUpdateNew()

def perform_evaluation(model, tokenizer, text, starting_point=1):
    model.eval()
    batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=text)
    with torch.no_grad():
        outs, pretokenized_score = model.forward_with_loss( #model.forward_with_loss_parallelized_efficient( #model.forward_with_loss_parallelized_efficient( #model.forward_with_loss_parallelized(
            batch_inputs=batch_inputs,
            compute_accuracy=True,
            return_dict=True
        )

    # memo_input = tokenizer.get_text_batch_encoding(text=text)
    # with torch.no_grad():
    #     pretokenized_score = evaluation.check_pretokenized(
    #         model=model,
    #         tokenizer=tokenizer,
    #         input_ids=memo_input['input_ids'],
    #         starting_point=starting_point
    #     )
    loss = outs['loss']
    del outs
    model.train()
    torch.cuda.empty_cache()
    return dict(
        out_loss=loss,
        token_accuracy=pretokenized_score
    )

Reading the two texts

In [3]:
with open("testo_di_prova.txt") as my_first_text_f:
    my_first_text = ''.join(my_first_text_f.read().split()[:50])
with open("testo_di_prova2.txt") as my_first_text_f:
    my_second_text = my_first_text_f.read()

Memorizing the first text and evaluating if it is memorized

In [4]:
memo_input_1 = tokenizer.get_text_batch_encoding([my_first_text]*1)  # Writing the same doc 8 times to stress the memorization with batch
memo_input_2 = tokenizer.get_text_batch_encoding([my_second_text]*1) # Writing the same doc 8 times to stress the memorization with batch

model.memorize_text(memo_input_1)

e_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
e_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)

print(f"Memorization level of first text: {round(e_1['token_accuracy']['accuracy'], 4)}")
print(f"Memorization level of second text: {round(e_2['token_accuracy']['accuracy'], 4)}")


Memorization level of first text: 0.0
Memorization level of second text: 0.0


Memorizing the second text and checking if it affected the memorization of the first text

In [5]:
model.memorize_text(memo_input_2)

e_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
e_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)

print(f"Memorization level of first text: {round(e_1['token_accuracy']['accuracy'], 4)}")
print(f"Memorization level of second text: {round(e_2['token_accuracy']['accuracy'], 4)}")

Memorization level of first text: 0.0
Memorization level of second text: 0.0


Forgetting the first document

In [6]:
model.forget_text(memo_input_2)

Checking the effect on the two texts

In [7]:
e_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
e_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)

print(f"Memorization level of first text: {round(e_1['token_accuracy']['accuracy'], 4)}")
print(f"Memorization level of second text: {round(e_2['token_accuracy']['accuracy'], 4)}")

Memorization level of first text: 0.0
Memorization level of second text: 0.0


In [8]:
exit()